# Cấp 6 – RL Self-Play Actor-Critic & RQ3 trên Google Colab GPU T4

Phục vụ **Issue #11** và **Proposal mục 2, 4, 5** (Nhóm 31 - TTNTNC):
- **Khởi tạo:** Nạp trọng số từ mạng Cấp 5 bản chuẩn (**epoch 54**, `/content/drive/MyDrive/caro/level5/best.pt`).
- **Thuật toán (đã ổn định hóa):** Actor-Critic với `lr = 1e-5`, chuẩn hóa advantage theo minibatch, cân sức hai bên (`temperature = opponent_temperature = 1.0`), và phạt `KL(pi || pi_SL)` (`kl_coef = 0.05` hoặc `0.1`) để giữ policy không trôi xa Cấp 5.
- **Thử nghiệm nhanh A/B/C (5 vòng mỗi cấu hình):** So sánh nhanh `lr` và `kl_coef` trước khi chạy đủ 20 vòng.
- **Tự động chạy tiếp (Auto-resume):** Lưu mọi checkpoint (`best.pt`, `latest.pt`, `rounds/round_*.pt`, `pool/`) và `train_log.jsonl` lên Google Drive. **Lưu ý:** dùng thư mục mới `MyDrive/caro/level6_v2` để giữ nguyên `train_log.jsonl` lần chạy đầu (`0.50 -> 0.07`) làm minh chứng báo cáo!


## 1. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Tải mã nguồn và cài đặt thư viện

In [ ]:
%cd /content
!test -d TTNTNC-NHOM31 || git clone https://github.com/minhhieu310800-jpg/TTNTNC-NHOM31.git
%cd /content/TTNTNC-NHOM31
# Nếu đang thử trước khi PR được merge vào main, đổi BRANCH thành 'caro/11-rl-tuning'
BRANCH = 'main'
!git fetch origin
!git checkout {BRANCH}
!git pull --ff-only
%cd do-an-caro
!pip install -q -r requirements.txt


## 3. Kiểm tra GPU T4 & Checkpoint Cấp 5 (đảm bảo đúng bản Epoch 54)

Ô này luôn ưu tiên chép bản mới nhất từ Drive `/content/drive/MyDrive/caro/level5/best.pt` và in `epoch` + `val_metrics` để chắc chắn đúng bản **epoch 54** (top1 ~69.3%), không dùng nhầm bản epoch 29 cũ.


In [ ]:
import shutil
from pathlib import Path
import torch

print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

DRIVE_L5 = Path('/content/drive/MyDrive/caro/level5/best.pt')
LOCAL_L5 = Path('checkpoints/level5/best.pt')
LOCAL_L5.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_L5.exists():
    shutil.copy2(DRIVE_L5, LOCAL_L5)
    print(f'Đã sao chép checkpoint Cấp 5 từ Drive về {LOCAL_L5}')
elif LOCAL_L5.exists():
    print(f'Đang dùng checkpoint Cấp 5 cục bộ: {LOCAL_L5}')
else:
    raise FileNotFoundError(
        f'Không tìm thấy checkpoint Cấp 5 tại {DRIVE_L5} hoặc {LOCAL_L5}.\n'
        'Vui lòng hoàn thành huấn luyện Cấp 5 (training/supervised-policy-value-network-colab.ipynb) trước.'
    )

meta = torch.load(LOCAL_L5, map_location='cpu', weights_only=False)
print('Epoch Cấp 5:', meta.get('epoch'), '| val_metrics:', meta.get('val_metrics'))


## 3.5. Thử nhanh 3 cấu hình A / B / C (mỗi cấu hình 5 vòng = 5.000 episodes)

Mỗi cấu hình chạy trong thư mục riêng (`level6_trial_A`, `level6_trial_B`, `level6_trial_C`) và dùng `--no-resume` trên chính thư mục thử đó để không ảnh hưởng tới `/content/drive/MyDrive/caro/level6` cũ.
- **A:** `lr=1e-5`, `kl_coef=0.05`, `opponent_temperature=1.0`
- **B:** `lr=1e-5`, `kl_coef=0.1`, `opponent_temperature=1.0`
- **C:** `lr=3e-5`, `kl_coef=0.1`, `opponent_temperature=1.0`


In [ ]:
TRIALS = [
    ('A', 1e-5, 0.05),
    ('B', 1e-5, 0.10),
    ('C', 3e-5, 0.10),
]
TRIAL_BASE = '/content/drive/MyDrive/caro'

for tag, lr_val, kl_val in TRIALS:
    trial_dir = f'{TRIAL_BASE}/level6_trial_{tag}'
    print(f'\n=== CHẠY THỬ {tag}: lr={lr_val}, kl_coef={kl_val}, opp_temp=1.0 -> {trial_dir} ===')
    !python -m training.train_rl configs/training/rl.yaml \
        --init-checkpoint checkpoints/level5/best.pt \
        --out-dir {trial_dir} \
        --device cuda \
        --episodes 5000 \
        --lr {lr_val} \
        --kl-coef {kl_val} \
        --opponent-temperature 1.0 \
        --no-resume


In [ ]:
import json
from pathlib import Path

print(f"{'Trial':>5} | {'BestRnd':>7} | {'BestScore':>9} | {'LastScore':>9} | {'LastLen':>7} | {'LastKL':>7} | {'LastEnt':>7}")
print('-' * 68)
for tag in ('A', 'B', 'C'):
    log_f = Path(f'/content/drive/MyDrive/caro/level6_trial_{tag}/train_log.jsonl')
    if not log_f.exists():
        continue
    recs = [json.loads(l) for l in log_f.read_text(encoding='utf-8').splitlines() if l.strip()]
    best_r = max(recs, key=lambda r: r['score_vs_sl'])
    last = recs[-1]
    print(
        f"{tag:>5} | {best_r['round']:7d} | {best_r['score_vs_sl']:9.3f} | "
        f"{last['score_vs_sl']:9.3f} | {last.get('mean_game_len', 0):7.2f} | "
        f"{str(last.get('kl') or '-'):>7} | {str(last.get('entropy') or '-'):>7}"
    )


## 4. Huấn luyện chính thức RL Self-Play 20 vòng (20.000 episodes)

1. Điền `CHOSEN_LR` và `CHOSEN_KL` từ cấu hình A/B/C tốt nhất (giữ được `score_vs_sl >= 0.5` và `mean_game_len` không tụt).
2. Mặc định lưu vào `OUT = '/content/drive/MyDrive/caro/level6_v2'` để **giữ nguyên `train_log.jsonl` của lần chạy đầu (`0.50 -> 0.07`)** trong `MyDrive/caro/level6` làm minh chứng báo cáo.
3. **Cảnh báo `--no-resume`:** Ô này cố ý KHÔNG bật `--no-resume` để tự động chạy tiếp nếu Colab ngắt kết nối. Chỉ thêm `--no-resume` khi bạn CHẮC CHẮN muốn xóa sạch `pool/`, `rounds/`, `best.pt`, `latest.pt` trong `{OUT}` để train lại từ đầu!
4. Tùy chọn: đặt `VALUE_WARMUP = 1` (hoặc `2`) nếu muốn chỉ train đầu value 1–2 vòng đầu trước khi mở cập nhật policy.


In [ ]:
OUT = '/content/drive/MyDrive/caro/level6_v2'
CHOSEN_LR = 1e-5       # chỉnh theo kết quả thử A/B/C
CHOSEN_KL = 0.05       # chỉnh theo kết quả thử A/B/C (0.05 hoặc 0.10)
VALUE_WARMUP = 0       # 0 = tắt; đặt 1 hoặc 2 nếu muốn khởi ấm đầu value trước

!python -m training.train_rl configs/training/rl.yaml \
    --init-checkpoint checkpoints/level5/best.pt \
    --out-dir {OUT} \
    --device cuda \
    --episodes 20000 \
    --lr {CHOSEN_LR} \
    --kl-coef {CHOSEN_KL} \
    --opponent-temperature 1.0 \
    --value-warmup-rounds {VALUE_WARMUP} \
    --max-hours 3.5


## 5. Vẽ đường cong học tập & Báo cáo RQ3 (SL vs RL Self-Play)

In [ ]:
from IPython.display import Image, display

FIG_PATH = f'{OUT}/rq3_learning_curve.png'
REPORT_PATH = f'{OUT}/rq3_report.md'
!python -m analysis.rq3_learning_curve --train-log {OUT}/train_log.jsonl --fig {FIG_PATH} --out {REPORT_PATH}
display(Image(filename=FIG_PATH))


## 6. Chạy giải đấu chính thức RQ3 (`configs/arena/rq3.yaml`)

Sao chép `best.pt`, `rounds/`, `train_log.jsonl` của lần chạy mới về `checkpoints/level6` rồi đấu đủ 200 ván/cặp trên 100 khai cuộc cân bằng (Mode B) với `sl` và `heuristic`, sau đó vẽ biểu đồ tổng hợp.


In [ ]:
import shutil
shutil.copytree(OUT, 'checkpoints/level6', dirs_exist_ok=True)
!python -m arena.tournament configs/arena/rq3.yaml
!python -m analysis.rq3_learning_curve \
    --train-log checkpoints/level6/train_log.jsonl \
    --arena-log logs/rq3_rl_vs_sl.jsonl \
    --fig {OUT}/rq3_arena_curve.png \
    --out {OUT}/rq3_arena_report.md
display(Image(filename=f'{OUT}/rq3_arena_curve.png'))


## 7. Thử nghiệm đối đầu nhanh: `RLAgent` (Cấp 6) vs `SLAgent` (Cấp 5)

In [ ]:
from game.game import CaroGame
from agents import RLAgent, SLAgent

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
rl_agent = RLAgent(checkpoint=f'{OUT}/best.pt', device=dev)
sl_agent = SLAgent(checkpoint='checkpoints/level5/best.pt', device=dev)

game = CaroGame()
while not game.is_terminal():
    if game.current_player == 1:
        action = rl_agent.select_action(game)
        tag, stats = 'RL (X)', rl_agent.last_stats
    else:
        action = sl_agent.select_action(game)
        tag, stats = 'SL (O)', sl_agent.last_stats
    r, c = divmod(action, 11)
    print(f"Nước {game.move_count + 1:2d}: {tag} -> ({r:2d}, {c:2d}) [P={stats['policy_prob']:.3f}, V={stats['value']:+.3f}]")
    game.step(action)

res = game.result()
print('Kết quả:', 'RL (X) thắng' if res == 1 else 'SL (O) thắng' if res == -1 else 'Hòa')
